<a href="https://colab.research.google.com/github/marcusvbrangel/production-surveillance/blob/main/production_surveillance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
# FASE 01 — LEITURA E QA/QC

Aprender:

- parse temporal
- índices temporais
- ordenação temporal
- missing data
- consistência temporal
---

In [53]:
# importacao das bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [54]:
# caminho do arquivo
arquivo = "/content/volve-field-daily-data.xlsx"

In [55]:
# leitura do arquivo excel
df_raw = pd.read_excel(arquivo)

In [56]:
# visualizacao inicial
df_raw.head()

,DATEPRD,WELL_BORE_CODE,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,NPD_FIELD_CODE,NPD_FIELD_NAME,NPD_FACILITY_CODE,NPD_FACILITY_NAME,ON_STREAM_HRS,AVG_DOWNHOLE_PRESSURE,...,AVG_CHOKE_UOM,AVG_WHP_P,AVG_WHT_P,DP_CHOKE_SIZE,BORE_OIL_VOL,BORE_GAS_VOL,BORE_WAT_VOL,BORE_WI_VOL,FLOW_KIND,WELL_TYPE
0,2014-04-07,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,0.00000,...,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,production,WI
1,2014-04-08,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,...,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,production,OP
2,2014-04-09,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,...,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,production,OP
3,2014-04-10,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,...,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,production,OP
4,2014-04-11,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,310.37614,...,%,33.09788,10.47992,33.07195,0.0,0.0,0.0,NaN,production,OP


In [57]:
# dimensao do dataset
df_raw.shape

(15634, 24)

In [58]:
# informacoes gerais: nome, nulos e tipos
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15634 entries, 0 to 15633
Data columns (total 24 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   DATEPRD                   15634 non-null  datetime64[ns]
 1   WELL_BORE_CODE            15634 non-null  object        
 2   NPD_WELL_BORE_CODE        15634 non-null  int64         
 3   NPD_WELL_BORE_NAME        15634 non-null  object        
 4   NPD_FIELD_CODE            15634 non-null  int64         
 5   NPD_FIELD_NAME            15634 non-null  object        
 6   NPD_FACILITY_CODE         15634 non-null  int64         
 7   NPD_FACILITY_NAME         15634 non-null  object        
 8   ON_STREAM_HRS             15349 non-null  float64       
 9   AVG_DOWNHOLE_PRESSURE     8980 non-null   float64       
 10  AVG_DOWNHOLE_TEMPERATURE  8980 non-null   float64       
 11  AVG_DP_TUBING             8980 non-null   float64       
 12  AVG_ANNULUS_PRESS 

In [59]:
# ============================================================
# 1. Parse temporal
# ============================================================

# converter a coluna de data para o tipo datetime
df_raw["DATEPRD"] = pd.to_datetime(df_raw["DATEPRD"], errors="coerce")

In [60]:
# verificacao do intervalo temporal geral
print("Data inicial:", df_raw["DATEPRD"].min())
print("Data final:", df_raw["DATEPRD"].max())

Data inicial: 2007-09-01 00:00:00
Data final: 2016-12-01 00:00:00


In [61]:
# verificacao se houve erro na conversao de data
datas_invalidas = df_raw["DATEPRD"].isna().sum()

print("Quantidade de datas invalidas:", datas_invalidas)

Quantidade de datas invalidas: 0


In [62]:
# ============================================================
# 2. Ordenação temporal
# ============================================================

# ordenar por poco e data
df_raw = df_raw.sort_values(
    by=["NPD_WELL_BORE_NAME", "DATEPRD"]
).reset_index(drop=True)

df_raw.head()

,DATEPRD,WELL_BORE_CODE,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,NPD_FIELD_CODE,NPD_FIELD_NAME,NPD_FACILITY_CODE,NPD_FACILITY_NAME,ON_STREAM_HRS,AVG_DOWNHOLE_PRESSURE,...,AVG_CHOKE_UOM,AVG_WHP_P,AVG_WHT_P,DP_CHOKE_SIZE,BORE_OIL_VOL,BORE_GAS_VOL,BORE_WAT_VOL,BORE_WI_VOL,FLOW_KIND,WELL_TYPE
0,2014-04-07,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,0.00000,...,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,production,WI
1,2014-04-08,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,...,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,production,OP
2,2014-04-09,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,...,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,production,OP
3,2014-04-10,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,...,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,production,OP
4,2014-04-11,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,310.37614,...,%,33.09788,10.47992,33.07195,0.0,0.0,0.0,NaN,production,OP


In [63]:
# verificacao dos pocos existente
df_raw["NPD_WELL_BORE_NAME"].unique()

array(['15/9-F-1 C', '15/9-F-11', '15/9-F-12', '15/9-F-14', '15/9-F-15 D',
       '15/9-F-4', '15/9-F-5'], dtype=object)

In [64]:
# quantidade de registros por poco
df_raw["NPD_WELL_BORE_NAME"].value_counts()

,count
NPD_WELL_BORE_NAME,
15/9-F-4,3327
15/9-F-5,3306
15/9-F-12,3056
15/9-F-14,3056
15/9-F-11,1165
15/9-F-15 D,978
15/9-F-1 C,746


In [65]:
# intervalo temporal por poco
intervalo_por_poco = (
    df_raw.groupby("NPD_WELL_BORE_NAME")
        .agg(
            data_inicial=("DATEPRD", "min"),
            data_final=("DATEPRD", "max"),
            qtd_registros=("DATEPRD", "count")
        )
        .sort_values("data_inicial")
)

intervalo_por_poco

,data_inicial,data_final,qtd_registros
NPD_WELL_BORE_NAME,,,
15/9-F-4,2007-09-01,2016-12-01,3327
15/9-F-5,2007-09-01,2016-09-18,3306
15/9-F-12,2008-02-12,2016-09-17,3056
15/9-F-14,2008-02-12,2016-09-17,3056
15/9-F-11,2013-07-08,2016-09-17,1165
15/9-F-15 D,2014-01-12,2016-09-17,978
15/9-F-1 C,2014-04-07,2016-04-21,746


In [66]:
# ============================================================
# 3. Missing data
# ============================================================

missing = (
    df_raw.isna()
    .sum()
    .to_frame("qtd_missing")
)

In [67]:
missing["percent_missing"] = (
    missing["qtd_missing"] / len(df_raw) * 100
).round(2)

missing.sort_values("percent_missing", ascending=False)

,qtd_missing,percent_missing
BORE_WI_VOL,9928,63.50
AVG_ANNULUS_PRESS,7744,49.53
AVG_CHOKE_SIZE_P,6715,42.95
AVG_DOWNHOLE_PRESSURE,6654,42.56
AVG_DOWNHOLE_TEMPERATURE,6654,42.56
AVG_DP_TUBING,6654,42.56
AVG_WHT_P,6488,41.50
AVG_WHP_P,6479,41.44
BORE_WAT_VOL,6473,41.40
AVG_CHOKE_UOM,6473,41.40


In [68]:
# missing por poco nas principais variaveis de producao
colunas_principais = [
    "ON_STREAM_HRS",
    "AVG_DOWNHOLE_PRESSURE",
    "AVG_DOWNHOLE_TEMPERATURE",
    "AVG_CHOKE_SIZE_P",
    "AVG_WHP_P",
    "AVG_WHT_P",
    "BORE_OIL_VOL",
    "BORE_GAS_VOL",
    "BORE_WAT_VOL",
]

In [69]:
missing_por_poco = (
    df_raw.groupby("NPD_WELL_BORE_NAME")[colunas_principais]
        .apply(lambda x: x.isna().mean() * 100)
        .round(2)
)

missing_por_poco

,ON_STREAM_HRS,AVG_DOWNHOLE_PRESSURE,AVG_DOWNHOLE_TEMPERATURE,AVG_CHOKE_SIZE_P,AVG_WHP_P,AVG_WHT_P,BORE_OIL_VOL,BORE_GAS_VOL,BORE_WAT_VOL
NPD_WELL_BORE_NAME,,,,,,,,,
15/9-F-1 C,0.00,0.40,0.40,0.00,0.00,0.00,0.00,0.00,0.00
15/9-F-11,0.00,0.52,0.52,0.17,0.52,0.52,0.00,0.00,0.00
15/9-F-12,0.00,0.20,0.20,1.44,0.00,0.00,0.00,0.00,0.00
15/9-F-14,0.00,0.20,0.20,6.41,0.00,0.00,0.00,0.00,0.00
15/9-F-15 D,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
15/9-F-4,4.57,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
15/9-F-5,4.02,100.00,100.00,95.16,95.16,95.43,95.16,95.16,95.16


In [70]:
# ============================================================
# 4. Consistência temporal
# ============================================================

# verificar datas duplicadas por poco
duplicadas = (
    df_raw.duplicated(subset=["NPD_WELL_BORE_NAME", "DATEPRD"])
        .sum()
)

print("Quantidade de registros duplicados por poco/data:", duplicadas)

Quantidade de registros duplicados por poco/data: 0


In [71]:
# mostrar duplicadas, se existirem
df_raw[df_raw.duplicated(subset=["NPD_WELL_BORE_NAME", "DATEPRD"], keep=False)]
#

,DATEPRD,WELL_BORE_CODE,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,NPD_FIELD_CODE,NPD_FIELD_NAME,NPD_FACILITY_CODE,NPD_FACILITY_NAME,ON_STREAM_HRS,AVG_DOWNHOLE_PRESSURE,...,AVG_CHOKE_UOM,AVG_WHP_P,AVG_WHT_P,DP_CHOKE_SIZE,BORE_OIL_VOL,BORE_GAS_VOL,BORE_WAT_VOL,BORE_WI_VOL,FLOW_KIND,WELL_TYPE


In [72]:
# verificar gaps temporais por poco
# como o dataset e diario, esperamos diferenca de 1 dia entre os registros consecutivos
df_raw["diff_dias"] = (
    df_raw.groupby("NPD_WELL_BORE_NAME")["DATEPRD"]
        .diff()
        .dt.days
)

gaps = df_raw[df_raw["diff_dias"] > 1][
        ["NPD_WELL_BORE_NAME", "DATEPRD", "diff_dias"]
    ]

gaps.head(20)

,NPD_WELL_BORE_NAME,DATEPRD,diff_dias
819,15/9-F-11,2013-09-20,2.0
951,15/9-F-11,2014-01-31,2.0
958,15/9-F-11,2014-02-08,2.0
1987,15/9-F-12,2008-04-29,2.0
2087,15/9-F-12,2008-08-08,2.0
2357,15/9-F-12,2009-05-06,2.0
2446,15/9-F-12,2009-08-04,2.0
2474,15/9-F-12,2009-09-02,2.0
2653,15/9-F-12,2010-03-02,3.0
2658,15/9-F-12,2010-03-08,2.0


In [73]:
# resumo de gaps por poco
resumo_gaps = (
    gaps.groupby("NPD_WELL_BORE_NAME")
        .agg(
            qtd_gaps=("diff_dias", "count"),
            maior_gap_dias=("diff_dias", "max"),
            media_gap_dias=("diff_dias", "mean")
        )
        .round(2)
)

resumo_gaps

,qtd_gaps,maior_gap_dias,media_gap_dias
NPD_WELL_BORE_NAME,,,
15/9-F-11,3,2.0,2.00
15/9-F-12,46,12.0,2.85
15/9-F-14,46,12.0,2.85
15/9-F-15 D,2,2.0,2.00
15/9-F-4,2,30.0,27.50


In [74]:
# ============================================================
# 5. Consistência básica dos valores
# ============================================================

# estatistica descritiva das variaveis numericas
df_raw.describe().T

,count,mean,min,25%,50%,75%,max,std
DATEPRD,15634,2012-11-07 17:39:58.004349440,2007-09-01 00:00:00,2010-07-30 00:00:00,2013-05-08 00:00:00,2015-02-19 00:00:00,2016-12-01 00:00:00,NaN
NPD_WELL_BORE_CODE,15634.0,5908.581745,5351.0,5599.0,5693.0,5769.0,7405.0,649.231622
NPD_FIELD_CODE,15634.0,3420717.0,3420717.0,3420717.0,3420717.0,3420717.0,3420717.0,0.0
NPD_FACILITY_CODE,15634.0,369304.0,369304.0,369304.0,369304.0,369304.0,369304.0,0.0
ON_STREAM_HRS,15349.0,19.994093,0.0,24.0,24.0,24.0,25.0,8.369978
AVG_DOWNHOLE_PRESSURE,8980.0,181.803869,0.0,0.0,232.896939,255.401455,397.58855,109.712363
AVG_DOWNHOLE_TEMPERATURE,8980.0,77.162969,0.0,0.0,103.186689,106.276591,108.502178,45.657948
AVG_DP_TUBING,8980.0,154.028787,0.0,83.665361,175.588861,204.319964,345.90677,76.752373
AVG_ANNULUS_PRESS,7890.0,14.8561,0.0,10.841437,16.308598,21.306125,30.019828,8.406822
AVG_CHOKE_SIZE_P,8919.0,55.168533,0.0,18.952989,52.096877,99.924288,100.0,36.692924


In [75]:
# verificar valores negativo em variaveis que, em tese, nao deveriam ser negativas
coluna_nao_negativas = [
    "ON_STREAM_HRS",
    "AVG_CHOKE_SIZE_P",
    "BORE_OIL_VOL",
    "BORE_GAS_VOL",
    "BORE_WAT_VOL",
    "BORE_WI_VOL",
]

for col in coluna_nao_negativas:
    qtd_negativos = (df_raw[col] < 0).sum()
    print(f"{col}: {qtd_negativos} valores negativos")

ON_STREAM_HRS: 0 valores negativos
AVG_CHOKE_SIZE_P: 0 valores negativos
BORE_OIL_VOL: 0 valores negativos
BORE_GAS_VOL: 0 valores negativos
BORE_WAT_VOL: 4 valores negativos
BORE_WI_VOL: 0 valores negativos


In [76]:
# verificar producao com poco teoricamente offline
mask = ((df_raw["ON_STREAM_HRS"] == 0) &
       (
           (df_raw["BORE_OIL_VOL"] > 0) |
           (df_raw["BORE_GAS_VOL"] > 0) |
           (df_raw["BORE_WAT_VOL"] > 0)
       )
)

offline_com_producao = df_raw[mask]

offline_com_producao[
    ["DATEPRD", "NPD_WELL_BORE_NAME", "ON_STREAM_HRS", "BORE_OIL_VOL", "BORE_GAS_VOL", "BORE_WAT_VOL"]
].head(20)

,DATEPRD,NPD_WELL_BORE_NAME,ON_STREAM_HRS,BORE_OIL_VOL,BORE_GAS_VOL,BORE_WAT_VOL
1301,2015-01-17,15/9-F-11,0.0,1026.57,150773.50,461.02
3283,2011-12-26,15/9-F-12,0.0,0.00,7.09,9.71
6339,2011-12-26,15/9-F-14,0.0,0.00,56.38,45.12


In [77]:
# verificar linhas com producao de oleo positiva, mas com horas online zeradas ou ausentes
mask = (
    (df_raw["BORE_OIL_VOL"] > 0) &
    (
        (df_raw["ON_STREAM_HRS"] <= 0) |
        (df_raw["ON_STREAM_HRS"].isna())
    )
)

problema_horas = df_raw[mask]

problema_horas[
    ["DATEPRD", "NPD_WELL_BORE_NAME", "ON_STREAM_HRS", "BORE_OIL_VOL"]
].head(20)

,DATEPRD,NPD_WELL_BORE_NAME,ON_STREAM_HRS,BORE_OIL_VOL
1301,2015-01-17,15/9-F-11,0.0,1026.57


In [78]:
# ============================================================
# 6. Criar índice temporal para análises futuras
# ============================================================

# guardar uma versao com o indice temporal
df_time = df_raw.set_index("DATEPRD").copy()

# Ordenar o DataFrame pelo indice para garantir a monotonicidade
df_time = df_time.sort_index()

df_time.head()

,WELL_BORE_CODE,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,NPD_FIELD_CODE,NPD_FIELD_NAME,NPD_FACILITY_CODE,NPD_FACILITY_NAME,ON_STREAM_HRS,AVG_DOWNHOLE_PRESSURE,AVG_DOWNHOLE_TEMPERATURE,...,AVG_WHP_P,AVG_WHT_P,DP_CHOKE_SIZE,BORE_OIL_VOL,BORE_GAS_VOL,BORE_WAT_VOL,BORE_WI_VOL,FLOW_KIND,WELL_TYPE,diff_dias
DATEPRD,,,,,,,,,,,,,,,,,,,,,
2007-09-01,NO 15/9-F-5 AH,5769,15/9-F-5,3420717,VOLVE,369304,MÆRSK INSPIRER,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,injection,WI,NaN
2007-09-01,NO 15/9-F-4 AH,5693,15/9-F-4,3420717,VOLVE,369304,MÆRSK INSPIRER,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,injection,WI,NaN
2007-09-02,NO 15/9-F-5 AH,5769,15/9-F-5,3420717,VOLVE,369304,MÆRSK INSPIRER,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,injection,WI,1.0
2007-09-02,NO 15/9-F-4 AH,5693,15/9-F-4,3420717,VOLVE,369304,MÆRSK INSPIRER,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,injection,WI,1.0
2007-09-03,NO 15/9-F-4 AH,5693,15/9-F-4,3420717,VOLVE,369304,MÆRSK INSPIRER,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,injection,WI,1.0


In [79]:
# teste: selecionar um periodo especifico
df_time.loc["2014":"2015"].head()

,WELL_BORE_CODE,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,NPD_FIELD_CODE,NPD_FIELD_NAME,NPD_FACILITY_CODE,NPD_FACILITY_NAME,ON_STREAM_HRS,AVG_DOWNHOLE_PRESSURE,AVG_DOWNHOLE_TEMPERATURE,...,AVG_WHP_P,AVG_WHT_P,DP_CHOKE_SIZE,BORE_OIL_VOL,BORE_GAS_VOL,BORE_WAT_VOL,BORE_WI_VOL,FLOW_KIND,WELL_TYPE,diff_dias
DATEPRD,,,,,,,,,,,,,,,,,,,,,
2014-01-01,NO 15/9-F-12 H,5599,15/9-F-12,3420717,VOLVE,369304,MÆRSK INSPIRER,24.0,0.000000,0.000000,...,32.843717,89.543534,4.211773,276.25,43268.98,4858.05,NaN,production,OP,1.0
2014-01-01,NO 15/9-F-11 H,7078,15/9-F-11,3420717,VOLVE,369304,MÆRSK INSPIRER,24.0,249.530852,106.207740,...,77.161721,59.570777,48.655474,1199.06,187807.44,166.35,NaN,production,OP,1.0
2014-01-01,NO 15/9-F-14 H,5351,15/9-F-14,3420717,VOLVE,369304,MÆRSK INSPIRER,24.0,253.116393,100.076937,...,32.547246,86.117947,3.333856,604.44,91618.13,3844.67,NaN,production,OP,1.0
2014-01-01,NO 15/9-F-5 AH,5769,15/9-F-5,3420717,VOLVE,369304,MÆRSK INSPIRER,24.0,NaN,NaN,...,NaN,NaN,0.000000,NaN,NaN,NaN,5850.050889,injection,WI,1.0
2014-01-01,NO 15/9-F-4 AH,5693,15/9-F-4,3420717,VOLVE,369304,MÆRSK INSPIRER,24.0,NaN,NaN,...,NaN,NaN,0.000000,NaN,NaN,NaN,5468.746109,injection,WI,1.0


In [80]:
df_time.shape

(15634, 24)

In [81]:
# ============================================================
# 7. Salvar dataset validado da Fase 01
# ============================================================

# resetar o index para a retornar a coluna "DATEPRD"
df_validado = df_time.reset_index().copy()

# opcional - remover a coluna auxiliar
df_validado = df_validado.drop(columns=["diff_dias"], errors="ignore").copy()

# salvar como csv limpo
df_validado.to_csv("volve_daily_fase01_validado.csv", index=False)

print("Arquivo salvo: volve_daily_fase01_validado.csv")

Arquivo salvo: volve_daily_fase01_validado.csv


Resumo:

Esse é o bloco base da Fase 01. Depois dele, a próxima etapa natural é criar a Fase 02 — EDA Temporal por poço, com gráficos, filtros, produção diária, água, gás, choke e pressão ao longo do tempo.

---
# FASE 02 — EDA TEMPORAL

#### Tópicos:

- resample
- rolling
- groupby temporal
- sazonalidade
- tendência
- correlação temporal

#### Aqui você vai começar a sentir:

- comportamento do poço
- tendência operacional
- instabilidade
- ruído
- sazonalidade operacional
- degradação de produção
- relação entre variáveis

Isso aqui já começa a parecer:

**Production Surveillance real**

#### OBJETIVO DA FASE 02

Construir uma EDA temporal profissional por poço.

Aprender:

- filtros temporais
- séries temporais
- rolling windows
- resample
- groupby temporal
- tendência
- sazonalidade
- correlação temporal
- comportamento operacional

### O QUE VAMOS ANALISAR?

Por poço:

- óleo
- gás
- água
- choke
- pressão fundo
- pressão cabeça
- temperatura
- horas online

#### IMPORTANTE

Nesta fase vamos:

- entender o comportamento
- entender o tempo
- entender o poço
---

In [82]:
# ============================================================
# leitura do dataset validado
# ===========================================================

df_eda = pd.read_csv("volve_daily_fase01_validado.csv")
df_eda.head()

,DATEPRD,WELL_BORE_CODE,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,NPD_FIELD_CODE,NPD_FIELD_NAME,NPD_FACILITY_CODE,NPD_FACILITY_NAME,ON_STREAM_HRS,AVG_DOWNHOLE_PRESSURE,...,AVG_CHOKE_UOM,AVG_WHP_P,AVG_WHT_P,DP_CHOKE_SIZE,BORE_OIL_VOL,BORE_GAS_VOL,BORE_WAT_VOL,BORE_WI_VOL,FLOW_KIND,WELL_TYPE
0,2007-09-01,NO 15/9-F-5 AH,5769,15/9-F-5,3420717,VOLVE,369304,MÆRSK INSPIRER,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,injection,WI
1,2007-09-01,NO 15/9-F-4 AH,5693,15/9-F-4,3420717,VOLVE,369304,MÆRSK INSPIRER,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,injection,WI
2,2007-09-02,NO 15/9-F-5 AH,5769,15/9-F-5,3420717,VOLVE,369304,MÆRSK INSPIRER,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,injection,WI
3,2007-09-02,NO 15/9-F-4 AH,5693,15/9-F-4,3420717,VOLVE,369304,MÆRSK INSPIRER,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,injection,WI
4,2007-09-03,NO 15/9-F-4 AH,5693,15/9-F-4,3420717,VOLVE,369304,MÆRSK INSPIRER,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,injection,WI


In [83]:
# converter o campo data para datetime
df_eda["DATEPRD"] = pd.to_datetime(df_eda["DATEPRD"])

In [84]:
# ordenar
df_eda = df_eda.sort_values(by=["NPD_WELL_BORE_NAME", "DATEPRD"])

In [85]:
# ============================================================
# ESCOLHER UM POÇO
# ============================================================

# IMPORTANTÍSSIMO.

# EDA temporal inicialmente:

# SEMPRE por poço
# ===========================================================

df_eda["NPD_WELL_BORE_NAME"].unique()

well_name = "15/9-F-1 C"

df_well = (
    df_eda[df_eda["NPD_WELL_BORE_NAME"] == well_name]
    .copy()
)

df_well.head()

,DATEPRD,WELL_BORE_CODE,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,NPD_FIELD_CODE,NPD_FIELD_NAME,NPD_FACILITY_CODE,NPD_FACILITY_NAME,ON_STREAM_HRS,AVG_DOWNHOLE_PRESSURE,...,AVG_CHOKE_UOM,AVG_WHP_P,AVG_WHT_P,DP_CHOKE_SIZE,BORE_OIL_VOL,BORE_GAS_VOL,BORE_WAT_VOL,BORE_WI_VOL,FLOW_KIND,WELL_TYPE
9496,2014-04-07,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,0.00000,...,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,production,WI
9508,2014-04-08,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,...,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,production,OP
9514,2014-04-09,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,...,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,production,OP
9521,2014-04-10,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,...,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,production,OP
9525,2014-04-11,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,310.37614,...,%,33.09788,10.47992,33.07195,0.0,0.0,0.0,NaN,production,OP


In [86]:
# verificar a dimensao
df_well.shape

(746, 24)

In [87]:
# definir indice temporal
df_well = df_well.set_index("DATEPRD")

In [88]:
df_well.head()

,WELL_BORE_CODE,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,NPD_FIELD_CODE,NPD_FIELD_NAME,NPD_FACILITY_CODE,NPD_FACILITY_NAME,ON_STREAM_HRS,AVG_DOWNHOLE_PRESSURE,AVG_DOWNHOLE_TEMPERATURE,...,AVG_CHOKE_UOM,AVG_WHP_P,AVG_WHT_P,DP_CHOKE_SIZE,BORE_OIL_VOL,BORE_GAS_VOL,BORE_WAT_VOL,BORE_WI_VOL,FLOW_KIND,WELL_TYPE
DATEPRD,,,,,,,,,,,,,,,,,,,,,
2014-04-07,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,0.00000,0.00000,...,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,production,WI
2014-04-08,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,NaN,...,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,production,OP
2014-04-09,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,NaN,...,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,production,OP
2014-04-10,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,NaN,...,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,production,OP
2014-04-11,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,310.37614,96.87589,...,%,33.09788,10.47992,33.07195,0.0,0.0,0.0,NaN,production,OP


---
# FASE 03 — FEATURE ENGINEERING TEMPORAL

Aprender:

- lag features
- rolling mean
- rolling std
- deltas
- cumulative features
- expanding windows
- EWMA
- diferenças percentuais
---

---
# FASE 04 — KPIs DE PRODUÇÃO

Criar:

- GOR
- Water Cut
- eficiência operacional
- drawdown proxies
- estabilidade operacional
---

---
# FASE 05 — DETECÇÃO DE ANOMALIAS

Detectar:

- shutdown
- comportamento estranho
- perda abrupta
- instabilidade
---

---
# FASE 06 — FORECASTING

Prever:

- produção
- water cut
- pressão
- tendência operacional
---